# GINからVG1GC-66のNWBを取得し、downsampled+ophysのみの軽量版にしてDriveへ保存

`docs/data.md` の実測どおり、GINのNWB（1ファイル約1.7GB）の内訳は `acquisition`（5kHz生波形16ch、約2.3GB論理サイズ、82%）、
`processing`（499MB、18%）。さらに`processing`は`behavior`（動画ネイティブレート、約302MB）・`downsampled`（30Hz、約130MB）・
`ophys`（30Hz、約65MB）に分かれる。

`bdbc_nwb_explorer.read_nwb()` はデフォルト(`downsampled=True`)では `processing['downsampled']` と `processing['ophys']` しか読まず、
`acquisition` と `processing['behavior']`（ネイティブレートのキーポイント）は一切参照しない
（`bdbc_nwb_explorer/view.py` の `read_acquisition()` / `read_video_tracking()` / `read_trials()` / `read_roi_dFF()` で確認済み）。
そのためこの2つを除去した軽量版（**約195MB**）でも既存パイプライン（`src/glmhmm_ver4.py`）は無改造で動く。

**このノートブックはWSLローカル実行専用**（Colabは使わない。理由は計画ファイル参照: GINのgit-annex転送はColabの一時ディスク・帯域と相性が悪く、無料枠のセッション時間制限もある）。

## 決定事項
- 対象: `VG1GC-66` の `task-day1`〜`task-day5`
- `acquisition` と `processing/behavior` を除去し、`processing/downsampled` と `processing/ophys` だけを残す
  （将来ネイティブレート約100Hzのキーポイントが必要になった場合はGINから再取得すればよい、という前提での破棄）
- 変換後、GIN由来のフル版NWBは**破棄**する（Driveにフル版は残さない）
- 既存の `nwb_manual/VG1GC-66/..._task-day15.nwb`（フル版）も今回軽量版化して統一する
- 再実行時、Driveに変換済みファイルが既にあるdayはスキップする（冪等）
- **1day処理するごとに、その日の生NWBを即座に削除**してから次のdayに進む（15日分を先にまとめて取得すると25GB超が一時的に溜まるため）


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

assert "WSL_DISTRO_NAME" in os.environ, (
    "このノートブックはWSLローカル実行を想定しています。"
    "Colab/Windowsネイティブでの実行は未対応です（datalad/git-annexが前提のため）。"
)

import config
import src.glmhmm_ver4 as v4
import src.nwb_shrink as shrink

print("DATA_NWB_ROOT:", config.DATA_NWB_ROOT)
print("DATA_NWB_ROOTが存在するか:", config.DATA_NWB_ROOT.exists())


## 依存関係の確認

`datalad` と `git-annex` が必要。**このノートブックからは自動インストールしない**（sudoが要るapt操作を伴うため）。
無ければ事前にWSLのターミナルで以下を実行しておく:

```bash
sudo apt-get update && sudo apt-get install -y git-annex
/mnt/c/Users/<user>/braidyn-bc/.venv-wsl/bin/pip install datalad
```


In [ ]:
for tool in ("datalad", "git-annex"):
    path = shutil.which(tool)
    status = path if path else "見つかりません"
    print(f"{tool}: {status}")


In [ ]:
MOUSE_ID = "VG1GC-66"
TARGET_DAYS = [f"task-day{n}" for n in range(1, 6)]  # day1〜day5

GIN_URL = "https://gin.g-node.org/BraiDyn-BC/Kondo2025_CuedLeverPullNWB"
# リポジトリ外の作業ディレクトリ(.gitignore不要)。datalad clone(メタデータのみ)を置く。
GIN_CACHE_DIR = Path.home() / "gin_cache" / "Kondo2025_CuedLeverPullNWB"
WORK_TMP_DIR = Path.home() / "gin_cache" / "_tmp_processing_only"

DEST_ROOT = config.DATA_NWB_ROOT / MOUSE_ID

print("GIN_CACHE_DIR:", GIN_CACHE_DIR)
print("WORK_TMP_DIR:", WORK_TMP_DIR)
print("DEST_ROOT:", DEST_ROOT)


## GINデータセットのclone（メタデータのみ）

`datalad clone` はファイル名・ディレクトリ構造だけを取得し、NWB本体（git-annexの実体）は取得しない。
既にcloneしてあれば再cloneしない。


In [ ]:
if not GIN_CACHE_DIR.exists():
    GIN_CACHE_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["datalad", "clone", GIN_URL, str(GIN_CACHE_DIR)], check=True)
else:
    print("既にcloneされています:", GIN_CACHE_DIR)


## 構造確認（GIN内の実際のパス・命名規則）

GIN側のディレクトリ構造（`VG1GC-66_..._task-dayN.nwb` のようにローカルの `nwb_manual` と同じ命名か、
BIDS風 `sub-XX/ses-XX` かなどは未確認。`src/nwb_shrink.find_gin_nwb_path()` は
mouse_id・task_dayを含むファイル名を再帰的に探すため、事前に厳密な構造を知らなくても動く想定だが、
**このセルを実行して実際に見つかるか必ず確認してから**次のセル（本体取得ループ）に進むこと。
見つからない場合はここで `gin_root.rglob("*.nwb")` の出力を見て、`find_gin_nwb_path` の
マッチ条件を実際の命名規則に合わせて調整する。


In [ ]:
print("GIN内の全NWBファイル数:", len(list(GIN_CACHE_DIR.rglob("*.nwb"))))
print("---")
for day in TARGET_DAYS:
    found = shrink.find_gin_nwb_path(GIN_CACHE_DIR, MOUSE_ID, day)
    print(f"{day}: {found}")


## day1〜day5を1日ずつ取得→変換→即削除

Driveに既にあるdayはスキップする。1day処理し終えるごとに、その日の生NWB（GINから取得した実体）を
`datalad drop` で即座に削除してから次のdayへ進む（ローカルのピーク使用量を1day分に抑えるため）。


In [ ]:
WORK_TMP_DIR.mkdir(parents=True, exist_ok=True)
DEST_ROOT.mkdir(parents=True, exist_ok=True)

summary = []

for day in TARGET_DAYS:
    print(f"\n=== {day} ===")

    existing = v4.find_nwb_file(MOUSE_ID, day)
    if existing is not None:
        print(f"既にDriveに存在するためスキップ: {existing}")
        summary.append((day, "skipped (already exists)", existing))
        continue

    src_path = shrink.find_gin_nwb_path(GIN_CACHE_DIR, MOUSE_ID, day)
    if src_path is None:
        print(f"GIN内で{MOUSE_ID} {day}のNWBが見つかりませんでした。スキップします。")
        summary.append((day, "skipped (not found on GIN)", None))
        continue

    print(f"GIN上のパス: {src_path}")
    print("datalad get 実行中...")
    subprocess.run(["datalad", "get", str(src_path)], check=True, cwd=str(GIN_CACHE_DIR))

    tmp_out = WORK_TMP_DIR / f"{day}_processing_only.nwb"
    if tmp_out.exists():
        tmp_out.unlink()

    try:
        print("acquisition・processing/behavior除去中...")
        shrink.strip_to_downsampled_and_ophys(src_path, tmp_out)

        print("読み込み検証中...")
        shrink.verify_processing_only(tmp_out)

        dest_name = shrink.normalized_dest_filename(src_path.name, MOUSE_ID, day)
        dest_path = DEST_ROOT / dest_name
        shutil.copy2(tmp_out, dest_path)
        print(f"Driveへ保存: {dest_path} ({dest_path.stat().st_size / 1e6:.1f} MB)")
        summary.append((day, "converted", dest_path))
    finally:
        # 生NWB・一時processing-only版を即削除してピーク使用量を抑える
        if tmp_out.exists():
            tmp_out.unlink()
        print("生NWBをdrop中...")
        subprocess.run(["datalad", "drop", str(src_path)], check=True, cwd=str(GIN_CACHE_DIR))

print("\n=== summary ===")
for day, status, path in summary:
    print(day, status, path)


## 既存day15の統一

`nwb_manual/VG1GC-66/..._task-day15.nwb`（フル版、GINから既に手動配置済みなのでGIN再取得は不要）を
processing-only版に置き換える。失敗時に元ファイルを失わないよう、一時退避してから安全に差し替える。


In [ ]:
DAY15 = "task-day15"
full_path = v4.find_nwb_file(MOUSE_ID, DAY15)
print("day15の現在のファイル:", full_path)

if full_path is not None:
    tmp_out = WORK_TMP_DIR / f"{DAY15}_processing_only.nwb"
    if tmp_out.exists():
        tmp_out.unlink()

    print("acquisition・processing/behavior除去中...")
    shrink.strip_to_downsampled_and_ophys(full_path, tmp_out)

    print("変換後ファイルの読み込み検証中...")
    shrink.verify_processing_only(tmp_out)

    print("差し替え中...")
    shrink.swap_in_place(tmp_out, full_path)

    print("差し替え後の読み込み再検証中...")
    shrink.verify_processing_only(full_path)

    shrink.finalize_swap(full_path)
    print(f"day15をprocessing-only化しました: {full_path} ({full_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("day15のファイルが見つからないため、何もしません。")


## 検証

- `nwb_manual/VG1GC-66/` 配下の全ファイルサイズ（フル版1.7GB程度より大幅に小さいはず）
- 各ファイルが `nwbx.read_nwb()` で読めて trials/imaging が空でないこと
- CSVがあるdayについては `v4.process_session()` が既存パイプラインとして動くこと


In [ ]:
for p in sorted(DEST_ROOT.glob("*.nwb")):
    size_mb = p.stat().st_size / 1e6
    try:
        shrink.verify_processing_only(p)
        status = "OK"
    except Exception as exc:
        status = f"NG: {exc}"
    print(f"{p.name}: {size_mb:.1f} MB  [{status}]")


In [ ]:
for day in TARGET_DAYS + [DAY15]:
    csv_dir = config.DATA_CSV_ROOT / MOUSE_ID / day
    if not (csv_dir / "trials_L1L2.csv").exists():
        print(f"{day}: CSVなし、スキップ")
        continue
    try:
        pack = v4.process_session(MOUSE_ID, day)
        print(f"{day}: process_session OK (n_trials={len(pack['trials'])}, has_face={pack['has_face']})")
    except Exception as exc:
        print(f"{day}: process_session NG: {exc}")
